# Vibration Isolation — As-Selected Re-Solve (COTS FC)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB5 (`vibration_isolation`) soft-mounted a *configured estimate* of the
FC/IMU cluster (`config/vibration.yaml`: 60 g). NB11 froze a real flight
controller. This notebook **re-solves the isolator sizing with the frozen
FC's catalog mass** — downstream, as a new notebook, keeping the pipeline
one-way (ADR-0012).

Because the 1-DOF isolator corner frequency depends only on the forcing
frequency, target, and damping — *not* on the isolated mass — the corner
frequency, transmissibility, static sag, and sway space are unchanged by
construction; what the real FC mass changes is the **isolator stiffness**
(k ∝ m) the parts must be procured to. The forcing side is also
re-anchored: the blade count now comes with a consistency check against
the frozen propeller's rating.

## Inputs

- `out/components.yaml` *(NB11 — frozen FC and propeller)*
- `out/vibration.yaml` *(NB5 — the conceptual solution, for the delta report)*
- `config/vibration.yaml`, `config/prop_geometry.yaml`, rotor RPM law

## Outputs

- `out/vibration_cots.yaml` *(same schema as `out/vibration.yaml` + frozen-FC
  block; consumed by NB14)*

---

In [1]:
import sys, math
from pathlib import Path
from dataclasses import replace
import yaml

REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))


# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (same pattern as NB2–NB12), derive
the EDF forcing frequency, and load the frozen hardware. The configured
blade count must match the frozen propeller — a mismatch means
`config/prop_geometry.yaml` and the COTS freeze have drifted apart.

---

In [2]:
from conceptual_design import (
    run_sizing_loop, FrozenComponents,
    Environment, Mission, Aerodynamics, Battery,
    WeightFraction, PropulsiveSystemParameters,
)
from conceptual_design.forward_flight_power import ForwardFlightParams
from conceptual_design.wing_sizing import WingStructureParams
from conceptual_design.models import RotorParams, Avionics
from conceptual_design.vtol_power import VTOLParams, rpm_from_diameter
from conceptual_design.vibration_isolation import (
    VibrationParams, size_isolation, write_vibration_yaml,
)

env     = Environment()
mission = Mission.from_yaml(CONFIG_PATH / "mission.yaml")
aero    = Aerodynamics.from_yaml(CONFIG_PATH / "aerodynamics.yaml")
batt    = Battery.from_yaml(CONFIG_PATH / "battery.yaml")
wf      = WeightFraction.from_yaml(CONFIG_PATH / "initial_weight_fraction_estimation.yaml")
prop    = PropulsiveSystemParameters.from_yaml(CONFIG_PATH / "propulsive_system_parameters.yaml")
ff      = ForwardFlightParams.from_yaml(CONFIG_PATH / "forward_flight_params.yaml")
ws      = WingStructureParams.from_yaml(CONFIG_PATH / "wing_structure_params.yaml")
rotor    = RotorParams.from_yaml(CONFIG_PATH / "rotor.yaml")
avionics = Avionics.from_yaml(CONFIG_PATH / "avionics.yaml")
vib_p    = VibrationParams.from_yaml(CONFIG_PATH / "vibration.yaml")

with open(CONFIG_PATH / "prop_geometry.yaml") as f:
    n_blades = int(yaml.safe_load(f)["n_blades"])

result = run_sizing_loop(
    m_payload_kg=mission.payload_kg, mission=mission, aero=aero, batt=batt,
    wf=wf, prop_params=prop, ff_params=ff, ws_params=ws, env=env,
    D_rotor_m=rotor.D_rotor_m, P_hotel_W=avionics.P_hotel_W,
)

vib_est = yaml.safe_load((OUT_PATH / "vibration.yaml").read_text(encoding="utf-8"))
comps   = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")
fc      = comps["flight_controller"]
prop_frozen = comps["propeller"]

assert int(prop_frozen.ratings["n_blades"]) == n_blades, (
    f"config/prop_geometry.yaml n_blades={n_blades} != frozen propeller "
    f"{prop_frozen.id} ({prop_frozen.ratings['n_blades']} blades) -- "
    "the configured rotor geometry has drifted from the COTS freeze")

vtol = VTOLParams.from_propulsive(prop, env)
rpm  = rpm_from_diameter(rotor.D_rotor_m, vtol)

print(f"EDF shaft speed   : {rpm:.0f} RPM  -> {rpm/60:.1f} Hz (1/rev)")
print(f"Blade count       : {n_blades}  (matches frozen {prop_frozen.id})")
print(f"Frozen FC         : {fc.name}  ({fc.id})")
print(f"  mass             : {fc.mass_kg*1e3:.1f} g   (NB5 estimate: {vib_p.m_fc_imu_kg*1e3:.1f} g)")
print(f"Payload (isolated): {result.m_payload_kg:.3f} kg  (unchanged -- not a COTS selection)")


EDF shaft speed   : 12211 RPM  -> 203.5 Hz (1/rev)
Blade count       : 3  (matches frozen ma_3blade_8x6)
Frozen FC         : Holybro Pixhawk 6C (plastic case)  (pixhawk_6c)
  mass             : 34.6 g   (NB5 estimate: 60.0 g)
Payload (isolated): 0.500 kg  (unchanged -- not a COTS selection)


# Section 2 — Re-Solve with the Frozen FC Mass

Same physics as NB5 (`size_isolation`, one call), with the configured
FC/IMU cluster estimate replaced by the frozen FC's catalog mass. Corner
frequency and window come out identical by construction; the FC-tray
isolator stiffness scales down with the lighter real board.

---

In [3]:
vib_p_cots = replace(vib_p, m_fc_imu_kg=fc.mass_kg)
res = size_isolation(rpm, n_blades, result.m_payload_kg, vib_p_cots)

lo, hi = res["f_n_window_hz"]
print(f"Forcing (1/rev)   : {res['f_shaft_hz']:.1f} Hz")
print(f"Corner freq f_n   : {res['f_n_hz']:.1f} Hz   "
      f"(valid window {lo:.0f}-{hi:.0f} Hz -> {'OK' if res['window_ok'] else 'OUT OF WINDOW'})")
print(f"Sway per bay      : {res['sway_mm']:.2f} mm  (unchanged -- mass-independent)")
print()
hdr = f"{'module':<10}{'m [kg]':>8}{'f_n [Hz]':>10}{'T':>8}{'d_st [mm]':>11}{'k/iso [N/m]':>13}{'hw [g]':>8}"
print(hdr); print('-' * len(hdr))
for nm, m in res["modules"].items():
    print(f"{nm:<10}{m.m_isolated_kg:>8.3f}{m.f_n_hz:>10.1f}"
          f"{m.transmissibility:>8.3f}{m.delta_static_mm:>11.3f}"
          f"{m.k_isolator_N_m:>13.0f}{m.hw_mass_kg*1e3:>8.0f}")

k_fc_est  = vib_est["modules"]["fc_imu"]["k_isolator_N_m"]
k_fc_cots = res["modules"]["fc_imu"].k_isolator_N_m
print()
print(f"FC-tray isolator stiffness: {k_fc_est:.0f} -> {k_fc_cots:.0f} N/m "
      f"({(k_fc_cots/k_fc_est - 1)*100:+.0f}% -- procure to this value)")

assert res["window_ok"], "corner frequency outside the valid isolation window"


Forcing (1/rev)   : 203.5 Hz
Corner freq f_n   : 55.7 Hz   (valid window 25-144 Hz -> OK)
Sway per bay      : 0.80 mm  (unchanged -- mass-independent)

module      m [kg]  f_n [Hz]       T  d_st [mm]  k/iso [N/m]  hw [g]
--------------------------------------------------------------------
fc_imu       0.035      55.7   0.100      0.080         1058      16
payload      0.500      55.7   0.100      0.080        15292      16

FC-tray isolator stiffness: 1835 -> 1058 N/m (-42% -- procure to this value)


# Section 3 — Output Export

`out/vibration_cots.yaml` — same schema as `out/vibration.yaml` (so the
fuselage re-solve consumes it identically) plus a `cots_fc` provenance
block.

---

In [4]:
write_vibration_yaml(
    res, vib_p_cots, OUT_PATH / "vibration_cots.yaml",
    regen_notebook="notebooks/vibration_isolation_cots.ipynb",
    extra={"cots_fc": {
        "id":     fc.id,
        "name":   fc.name,
        "mass_g": round(fc.mass_kg * 1e3, 2),
    }},
)
print(f"As-selected vibration design written -> {OUT_PATH / 'vibration_cots.yaml'}")


As-selected vibration design written -> D:\Dev\vbat-uav-notebooks\out\vibration_cots.yaml


# Section 4 — Delta vs. the Conceptual Solution (NB5) and Summary

---

In [5]:
fc_mod = res["modules"]["fc_imu"]
pl_mod = res["modules"]["payload"]

bar = "=" * 62
print(bar)
print("  VIBRATION AS-SELECTED RE-SOLVE SUMMARY".center(62))
print(bar)
print(f"  {'quantity':<32}{'NB5 (est.)':>12}{'this NB':>12}")
print("  " + "-" * 58)
print(f"  {'isolated FC/IMU mass [g]':<32}{vib_est['modules']['fc_imu']['m_isolated_kg']*1e3:>12.1f}{fc_mod.m_isolated_kg*1e3:>12.1f}")
print(f"  {'FC isolator k [N/m]':<32}{vib_est['modules']['fc_imu']['k_isolator_N_m']:>12.0f}{fc_mod.k_isolator_N_m:>12.0f}")
print(f"  {'corner frequency f_n [Hz]':<32}{vib_est['f_n_hz']:>12.1f}{res['f_n_hz']:>12.1f}")
print(f"  {'sway pad total [mm]':<32}{vib_est['sway_pad_total_m']*1e3:>12.2f}{2*res['sway_mm']:>12.2f}")
print()
print(f"  {'frozen FC':<32}: {fc.id}  ({fc.mass_kg*1e3:.1f} g)")
print(f"  {'1/rev attenuation':<32}: {fc_mod.attenuation_pct:.1f} %  (target met, window OK)")
print(f"  {'payload isolators':<32}: {pl_mod.n_isolators} x, k = {pl_mod.k_isolator_N_m:.0f} N/m (unchanged)")
print(bar)


             VIBRATION AS-SELECTED RE-SOLVE SUMMARY           
  quantity                          NB5 (est.)     this NB
  ----------------------------------------------------------
  isolated FC/IMU mass [g]                60.0        34.6
  FC isolator k [N/m]                     1835        1058
  corner frequency f_n [Hz]               55.7        55.7
  sway pad total [mm]                     1.60        1.60

  frozen FC                       : pixhawk_6c  (34.6 g)
  1/rev attenuation               : 90.0 %  (target met, window OK)
  payload isolators               : 4 x, k = 15292 N/m (unchanged)
